
# Experimentación - capas Medallion vs. tiempo de consulta

#### 1. Objetivo:
Comparar cuánto tarda y cuánto se lee al responder la misma pregunta desde Bronze, Silver y Gold.
#### 2. Requisito
Tener creadas las tablas `bronze_supply_chain`, `silver_supply_chain`, `gold_kpi_modo_envio` y `gold_region_categoria` 
#### 3. Importante
- Dos preguntas: P1 (% de entregas tardías por modo de envío) y P2 (entregas tardías por región y categoría).
- En Bronze y Silver se calcula la consulta; en Gold se lee la tabla ya calculada.
- Cada consulta se mide 5 veces, más una corrida de calentamiento que se descarta.
#### 4. Hipótesis
1. Gold será la más rápida y Bronze la más lenta.
2. Bronze y Silver tendrán tiempos parecidos.
3. La ventaja de Gold se notará más en P2 que en P1.
4. Las tres capas darán el mismo resultado.

In [0]:
# Preparación
import time
import pandas as pd
from pyspark.sql import functions as F

BRONZE, SILVER = "bronze_supply_chain", "silver_supply_chain"
GOLD_P1, GOLD_P2 = "gold_kpi_modo_envio", "gold_region_categoria"
N_REP = 5

bronze, silver = spark.table(BRONZE), spark.table(SILVER)

meta = {"fecha": time.strftime("%Y-%m-%d %H:%M:%S"), "spark": spark.version,
        "compute": "Databricks Free Edition (serverless)",
        "repeticiones": N_REP, "calentamiento": 1}
print(meta)

{'fecha': '2026-11-25 05:46:18', 'spark': '4.2.0', 'compute': 'Databricks Free Edition (serverless)', 'repeticiones': 5, 'calentamiento': 1}


In [0]:
#Resultados
def ficha(capa, nombre):
    t = spark.table(nombre)
    d = spark.sql(f"DESCRIBE DETAIL {nombre}").collect()[0]
    return {"capa": capa, "tabla": nombre, "filas": t.count(), "columnas": len(t.columns),
            "tamano_MB": round(d["sizeInBytes"] / 1024 / 1024, 2), "archivos": d["numFiles"]}

fichas = pd.DataFrame([ficha("Bronze", BRONZE), ficha("Silver", SILVER),
                       ficha("Gold (P1)", GOLD_P1), ficha("Gold (P2)", GOLD_P2)])
display(fichas)

capa,tabla,filas,columnas,tamano_MB,archivos
Bronze,bronze_supply_chain,180519,55,8.05,2
Silver,silver_supply_chain,180519,54,7.76,1
Gold (P1),gold_kpi_modo_envio,4,7,0.0,1
Gold (P2),gold_region_categoria,691,6,0.01,1


In [0]:
# P1: % de entregas tardías por modo de envío
def p1(t):
    return (t.groupBy("shipping_mode")
             .agg(F.count("*").alias("entregas"),
                  F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias")))

# P2: entregas tardías por región y categoría
def p2(t):
    return (t.groupBy("order_region", "category_name")
             .agg(F.sum("late_delivery_risk").alias("entregas_tardias")))

# En Gold no se calcula nada: se lee la tabla que ya existe
def p1_gold():
    return spark.table(GOLD_P1).select("shipping_mode", "entregas", "pct_tardias")

def p2_gold():
    return spark.table(GOLD_P2).select("order_region", "category_name", "entregas_tardias")

# Las 6 pruebas: (pregunta, capa, consulta)
PRUEBAS = [
    ("P1 modo de envío",      "Bronze", lambda: p1(bronze)),
    ("P1 modo de envío",      "Silver", lambda: p1(silver)),
    ("P1 modo de envío",      "Gold",   p1_gold),
    ("P2 región y categoría", "Bronze", lambda: p2(bronze)),
    ("P2 región y categoría", "Silver", lambda: p2(silver)),
    ("P2 región y categoría", "Gold",   p2_gold),
]
print("Pruebas definidas:", len(PRUEBAS))

Pruebas definidas: 6


In [0]:
# ¿Las tres capas responden lo mismo? (P1)
b = p1(bronze).withColumnRenamed("entregas", "ent_bronze").withColumnRenamed("pct_tardias", "pct_bronze")
s = p1(silver).withColumnRenamed("entregas", "ent_silver").withColumnRenamed("pct_tardias", "pct_silver")
g = p1_gold().withColumnRenamed("entregas", "ent_gold").withColumnRenamed("pct_tardias", "pct_gold")
display(b.join(s, how="inner", on="shipping_mode").join(g, how="inner", on="shipping_mode"))

# P2: cantidad de grupos y total de entregas tardías en cada capa
for capa, q in [("Bronze", p2(bronze)), ("Silver", p2(silver)), ("Gold", p2_gold())]:
    r = q.agg(F.count("*").alias("grupos"), F.sum("entregas_tardias").alias("total_tardias")).collect()[0]
    print(f"{capa:<7} grupos: {r['grupos']:>4} | total de tardías: {r['total_tardias']:,}")

shipping_mode,ent_bronze,pct_bronze,ent_silver,pct_silver,ent_gold,pct_gold
First Class,27814,95.32,27814,95.32,27814,95.32
Second Class,35216,76.63,35216,76.63,35216,76.63
Same Day,9737,45.74,9737,45.74,9737,45.74
Standard Class,107752,38.07,107752,38.07,107752,38.07


Bronze  grupos:  691 | total de tardías: 98,977
Silver  grupos:  691 | total de tardías: 98,977
Gold    grupos:  691 | total de tardías: 98,977


In [0]:
def cronometrar(consulta):
    t0 = time.perf_counter()
    consulta().collect()
    return time.perf_counter() - t0

registros = []
for pregunta, capa, consulta in PRUEBAS:
    cronometrar(consulta)                       # calentamiento (se descarta)
    for rep in range(1, N_REP + 1):
        registros.append({"pregunta": pregunta, "capa": capa, "repeticion": rep,
                          "segundos": round(cronometrar(consulta), 4)})

mediciones = pd.DataFrame(registros)
print("Listo:", len(mediciones), "mediciones")

Listo: 30 mediciones


In [0]:
# Resumen: mediana, variación (CV) y cuántas veces más rápida es cada capa que Bronze
res = (mediciones.groupby(["pregunta", "capa"])["segundos"]
         .agg(mediana="median", media="mean", desv="std").reset_index())
res["cv_pct"] = (res["desv"] / res["media"] * 100).round(1)

t_bronze = res[res.capa == "Bronze"].set_index("pregunta")["mediana"]
res["veces_mas_rapido_que_bronze"] = (res["pregunta"].map(t_bronze) / res["mediana"]).round(2)

orden = {"Bronze": 0, "Silver": 1, "Gold": 2}
res = (res.assign(o=res["capa"].map(orden)).sort_values(["pregunta", "o"])
          .drop(columns="o").round(3).reset_index(drop=True))
display(res)

pregunta,capa,mediana,media,desv,cv_pct,veces_mas_rapido_que_bronze
P1 modo de envío,Bronze,0.396,0.39,0.011,2.7,1.0
P1 modo de envío,Silver,0.321,0.326,0.019,5.7,1.23
P1 modo de envío,Gold,0.296,0.3,0.012,3.8,1.34
P2 región y categoría,Bronze,0.388,0.398,0.038,9.6,1.0
P2 región y categoría,Silver,0.335,0.346,0.032,9.1,1.16
P2 región y categoría,Gold,0.302,0.304,0.008,2.7,1.28


In [0]:
display(res[["capa","mediana","pregunta"]])

capa,mediana,pregunta
Bronze,0.396,P1 modo de envío
Silver,0.321,P1 modo de envío
Gold,0.296,P1 modo de envío
Bronze,0.388,P2 región y categoría
Silver,0.335,P2 región y categoría
Gold,0.302,P2 región y categoría


Databricks visualization. Run in Databricks to view.

In [0]:
display(fichas[["tabla", "filas"]])

tabla,filas
bronze_supply_chain,180519
silver_supply_chain,180519
gold_kpi_modo_envio,4
gold_region_categoria,691


Databricks visualization. Run in Databricks to view.

In [0]:
# Evidencia: guardar los resultados como tablas
def guardar(pdf, nombre):
    (spark.createDataFrame(pdf).write.mode("overwrite")
          .option("overwriteSchema", "true").saveAsTable(nombre))
    print("Guardado:", nombre)

guardar(fichas, "exp_capas_fichas")
guardar(mediciones, "exp_capas_mediciones")
guardar(res, "exp_capas_resumen")

Guardado: exp_capas_fichas
Guardado: exp_capas_mediciones
Guardado: exp_capas_resumen
